# 06 Fine-tuning pretrained multilingual Transformers (run on Google Colab with a GPU)

**Runtime → Change runtime type → T4 GPU.** The whole notebook takes about 45-60 minutes on a T4.

**Questions**
1. Does a pretrained Transformer beat the TF-IDF baseline on 608 Lingala articles?
2. Does it matter whether the model was adapted to African languages? We compare
   * `xlm-roberta-base` (Conneau et al., 2020): 100 languages from CommonCrawl, mostly high-resource; includes French and Swahili.
   * `Davlan/afro-xlmr-base` (Alabi et al., 2022): XLM-R further pre-trained (MAFT) on 17 African languages plus English, French and Arabic.
     **Lingala is not one of the 17**, so any gain over XLM-R comes from related Bantu languages (Swahili, Kinyarwanda, Kirundi, …) and French.
   * `Davlan/afro-xlmr-base-76L` (optional): a later MAFT model on 76 languages. Check its model card for Lingala (`lin`) before
     claiming that it saw Lingala in pre-training.
3. For the best model: head-only vs head+tail truncation, and class weights on/off.

**Fine-tuning recipe** (Devlin et al., 2019; Sun et al., 2019): AdamW, learning rate 2e-5, 10% linear warm-up then linear decay,
weight decay 0.01, batch size 16, up to 10 epochs, early stopping (patience 3) on validation macro-F1, gradient clipping 1.0,
mixed precision (fp16) on the GPU. The input is `headline. text`, up to 512 subword tokens.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Change this to your own GitHub repository before running in Colab
REPO_URL = "https://github.com/YOUR-GITHUB-USERNAME/lingala-news-classification.git"

if IN_COLAB and not Path("lingala-news-classification").exists():
    !git clone -q {REPO_URL}
if IN_COLAB:
    %cd lingala-news-classification
    %pip install -q transformers sentencepiece protobuf huggingface_hub

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import json
import shutil

import pandas as pd
import torch
from transformers import AutoTokenizer

from src import config, data_loading, evaluation, plots, transformer_training
from src.experiment_log import log_experiment

MODEL_FAMILY = "transformer_pretrained"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be very slow)")

train, validation, test = data_loading.load_all_splits()
validation_ids = data_loading.labels_to_ids(validation["category"])
test_ids = data_loading.labels_to_ids(test["category"])

CHECKPOINTS = {
    "xlmr_base": "xlm-roberta-base",
    "afroxlmr_base": "Davlan/afro-xlmr-base",
    "afroxlmr_base_76L": "Davlan/afro-xlmr-base-76L",  # optional, see the note above
}
RUN_OPTIONAL_76L = True

## How well does each tokeniser fit Lingala?
*Tokens per word* (fertility): a tokeniser that does not know a language splits each word into many short pieces, which
wastes the 512-token budget and makes the model's job harder (Rust et al., 2021).

In [ ]:
rows = []
for short_name, checkpoint in CHECKPOINTS.items():
    if short_name.endswith("76L") and not RUN_OPTIONAL_76L:
        continue
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    stats = transformer_training.subword_statistics(tokenizer, train[config.TEXT_COLUMN])
    rows.append({"model": short_name, **stats, "example": " ".join(tokenizer.tokenize("Bakonzi ya bokolongono basengi bato bamibatela"))})
fertility = pd.DataFrame(rows)
fertility.to_csv(config.RESULTS_DIRECTORY / "tokeniser_fertility.csv", index=False)
fertility

In [ ]:
BASE = {"learning_rate": 2e-5, "batch_size": 16, "epochs": 10, "patience": 3,
        "strategy": "head_tail", "max_tokens": 512, "class_weight": True, "seed": config.RANDOM_SEED}
results, histories = [], {}

def run(name, short_name, changes, notes, seed=config.RANDOM_SEED):
    settings = {**BASE, **changes, "checkpoint": CHECKPOINTS[short_name], "seed": seed}
    model, tokenizer, history = transformer_training.run_experiment(settings, train, validation)
    probabilities = transformer_training.predict_probabilities(model, tokenizer, validation, settings)
    scores = evaluation.compute_scores(validation_ids, probabilities)
    scores["best_epoch"] = history["best_epoch"]
    log_experiment(MODEL_FAMILY, name, "validation", settings, scores, notes)
    results.append({"experiment": name, "model": short_name, "change": notes,
                    "macro_f1": scores["macro_f1"], "weighted_f1": scores["weighted_f1"],
                    "log_loss": scores["log_loss"], "best_epoch": history["best_epoch"]})
    histories[name] = history
    plots.plot_learning_curves(history, "transformer_pretrained", name)
    del model
    transformer_training.free_memory()
    print(results[-1])

## Experiment group P: which pretrained model?

In [ ]:
run("P1_xlmr_base", "xlmr_base", {}, "XLM-R base: no African-language adaptation")
run("P2_afroxlmr_base", "afroxlmr_base", {}, "AfroXLMR base: adapted to 17 African languages (not Lingala)")
if RUN_OPTIONAL_76L:
    run("P3_afroxlmr_base_76L", "afroxlmr_base_76L", {}, "AfroXLMR base 76L: adapted to 76 languages")
pd.DataFrame(results)

## Experiment group Q: truncation and class weights (best model from group P)

In [ ]:
best_model = pd.DataFrame(results).sort_values(["macro_f1", "log_loss"], ascending=[False, True]).iloc[0]["model"]
print("Best pretrained model:", best_model)
run("Q1_head_only", best_model, {"strategy": "head"}, "Keep only the first 510 tokens")
run("Q2_no_class_weights", best_model, {"class_weight": False}, "Unweighted cross-entropy")
summary = pd.DataFrame(results)
summary.to_csv(config.RESULTS_DIRECTORY / "transformer_validation_summary.csv", index=False)
summary

## Final: best configuration with three seeds on the test set

In [ ]:
best_row = summary.sort_values(["macro_f1", "log_loss"], ascending=[False, True]).iloc[0]
final_changes = {"Q1_head_only": {"strategy": "head"}, "Q2_no_class_weights": {"class_weight": False}}.get(best_row["experiment"], {})
final_short_name = best_row["model"]
print("Final:", best_row["experiment"], final_short_name, final_changes)

seed_scores, first = [], None
for seed in config.FINAL_RUN_SEEDS:
    settings = {**BASE, **final_changes, "checkpoint": CHECKPOINTS[final_short_name], "seed": seed}
    model, tokenizer, history = transformer_training.run_experiment(settings, train, validation)
    probabilities = transformer_training.predict_probabilities(model, tokenizer, test, settings)
    seed_scores.append(evaluation.compute_scores(test_ids, probabilities))
    print(f"seed {seed}: test macro-F1 {seed_scores[-1]['macro_f1']:.4f}")
    if first is None:
        first = probabilities
        output = config.MODELS_DIRECTORY / "transformer_final"
        model.save_pretrained(output)
        tokenizer.save_pretrained(output)
        (output / "training_settings.json").write_text(json.dumps(settings, indent=2))
    del model
    transformer_training.free_memory()

summary_scores = evaluation.summarise_runs(seed_scores)
summary_scores["macro_f1_ci_low"], summary_scores["macro_f1_ci_high"] = evaluation.bootstrap_confidence_interval(test_ids, first.argmax(1))
log_experiment(MODEL_FAMILY, f"FINAL_{final_short_name}", "test", {**BASE, **final_changes, "checkpoint": CHECKPOINTS[final_short_name], "seeds": config.FINAL_RUN_SEEDS},
               summary_scores, "Mean/std over 3 seeds; CI and saved model from seed 42")
evaluation.save_test_predictions(first, "transformer")
print(f"Test macro-F1 {summary_scores['macro_f1']:.4f} ± {summary_scores['macro_f1_std']:.4f}; weighted-F1 {summary_scores['weighted_f1']:.4f}")
evaluation.print_classification_report(test_ids, first)
plots.plot_confusion_matrix(test_ids, first, "transformer", "test")
evaluation.save_misclassified_examples(test, test_ids, first, "transformer");

## Publish the model for the web app (Hugging Face Hub)
The fine-tuned model is about 1 GB, too big for GitHub. Upload it to your Hugging Face account; the Space then loads it by name.
Create a *write* token at https://huggingface.co/settings/tokens and paste it when asked.

In [ ]:
HF_MODEL_REPO = "YOUR-HF-USERNAME/lingala-news-topic-classifier"   # change this
PUSH_TO_HUB = False  # set to True after changing the name

if PUSH_TO_HUB:
    from huggingface_hub import login
    from transformers import AutoModelForSequenceClassification
    login()
    model = AutoModelForSequenceClassification.from_pretrained(config.MODELS_DIRECTORY / "transformer_final")
    tokenizer = AutoTokenizer.from_pretrained(config.MODELS_DIRECTORY / "transformer_final")
    model.push_to_hub(HF_MODEL_REPO)
    tokenizer.push_to_hub(HF_MODEL_REPO)
    print("Uploaded to https://huggingface.co/" + HF_MODEL_REPO)

## Bring the results back to your repository
Download this zip and unpack it over the `results/` folder of your local repository, then commit. Notebook 07 will then
include the Transformer in the comparison and the error analysis.

In [ ]:
archive = shutil.make_archive("transformer_results", "zip", root_dir=".", base_dir="results")
print("Created", archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)